# Documentazione di Progetto: Pipeline di Previsione Direzionale su Serie Storiche (XAUUSD)

## 1. Introduzione al Progetto
Il presente progetto ha come obiettivo lo sviluppo, il miglioramento e la validazione statistica di un algoritmo per la previsione della direzione della candela successiva in un mercato finanziario (asset: XAUUSD). 

Originariamente implementato in ambiente MQL4 tramite un approccio basato su *Pattern Matching* e *Cosine Similarity* (1-Nearest Neighbor), il sistema è in fase di migrazione e potenziamento in ambiente Python. L'obiettivo metodologico è trasformare un sistema di trading euristico in una pipeline di Data Science rigorosa, integrando tecniche avanzate di preprocessing, esplorazione statistica e modelli di Machine Learning (Classificazione e Regressione) per massimizzare la precisione predittiva. 

L'algoritmo analizza vettori di feature composti da sequenze storiche (es. 10 candele consecutive con valori OHLC) per prevedere la polarità (BUY/SELL) della candela successiva.

---

## 2. La Fase di Data Cleaning: Motivazioni e Risoluzione dei Problemi
La fase di preparazione dei dati (Data Cleaning e Preprocessing) rappresenta le fondamenta dell'intera pipeline analitica. I modelli di Machine Learning e le metriche di similarità vettoriale richiedono input rigorosamente strutturati, continui e numerici. 

I dati storici grezzi (estratti in questo caso da FXCM a timeframe 1 minuto, comprensivi di Bid e Ask) presentano nativamente diverse criticità strutturali e informative che renderebbero impossibile l'addestramento di un modello:
1. **Ambiguità dei Delimitatori:** I dataset europei o estratti da specifiche piattaforme utilizzano spesso la virgola (`,`) sia come separatore dei campi CSV, sia come separatore decimale per i prezzi. Questo genera un disallineamento strutturale delle matrici di dati.
2. **Tipi di Dato Errati:** Un prezzo formattato con la virgola viene interpretato dai parser statistici (come Pandas) come dato testuale (Stringa) anziché come numero a virgola mobile (Float), impedendo qualsiasi operazione matematica.
3. **Inconsistenze Temporali:** Presenza di gap dovuti alle chiusure dei mercati (weekend) o a tick mancanti, che distorcono la continuità delle finestre temporali utilizzate per i sample.

La presente sezione della pipeline si occupa della *Normalizzazione Strutturale* del dataset, garantendo che i dati grezzi vengano convertiti in un formato tabellare leggibile, coerente e tipizzato correttamente per le successive fasi di analisi statistica.

---

## 3. Pipeline di Normalizzazione Strutturale

### 3.1 Step 1: Risoluzione delle Anomalie Strutturali (`1Converter.py`)
Il primo script della pipeline affronta il problema critico dell'ambiguità dei delimitatori nel file CSV grezzo.

* **Problema:** A causa dell'utilizzo della virgola sia come separatore di colonna che come separatore decimale (es. `2062,14`), un parser standard divide un singolo valore di prezzo in due colonne distinte. Inoltre, i prezzi privi di decimali (es. `2062`) non subiscono questa divisione, generando righe con un numero variabile di campi e causando il fallimento sistematico dell'importazione dei dati.
* **Metodologia:** L'algoritmo `1Converter.py` opera riga per riga per ricostruire l'integrità strutturale. Isola le colonne temporali (Date, Time) e processa sequenzialmente i successivi valori. Sfruttando la lunghezza delle stringhe, l'algoritmo discrimina tra la parte intera di un prezzo e la sua potenziale parte decimale. Qualora un prezzo si presenti come numero intero puro, il sistema provvede all'imputazione di un campo fittizio (`'00'`) per ripristinare il corretto offset delle colonne.
* **Output:** Il risultato è un dataset intermedio (`1DataConverted.csv`) in cui ogni riga è forzata in modo deterministico a una lunghezza costante di 19 campi, separando provvisoriamente la radice intera dalla mantissa decimale.

In [ ]:
# Esecuzione Step 1: Risoluzione anomalie strutturali
%run "Data Management/DataCleaning/1Converter.py"

### 3.2 Step 2: Standardizzazione Numerica e Unificazione (`2MergeColumn.py`)
Il secondo script finalizza la formattazione strutturale, preparando i dati per l'ingestione in librerie di calcolo scientifico come `pandas` e `scikit-learn`.

* **Problema:** I dati in uscita dallo Step 1, pur essendo allineati, presentano i prezzi frammentati in due colonne e necessitano di essere convertiti nello standard internazionale (floating-point con separatore a punto) per abilitare le operazioni matematiche.
* **Metodologia:** L'algoritmo `2MergeColumn.py` acquisisce il file a 19 campi. Dopo aver instanziato un'intestazione pulita a 11 colonne (escludendo eventuali header corrotti del file precedente), itera sulle righe del dataset. Le coppie di colonne rappresentanti la parte intera e decimale vengono concatenate utilizzando il punto (`.`) come operatore di giunzione testuale (es. `2062` e `14` diventano `2062.14`). 
* **Output:** Il risultato è il dataset `2DataMerged.csv`, perfettamente ricompattato nella sua struttura originale a 11 colonne (Date, Time, 8 serie di prezzi OHLC Bid/Ask, TotalTicks). I dati sono ora pronti per essere elaborati algoritmicamente senza errori di parsing.

In [ ]:
# Esecuzione Step 2: Merge delle colonne
%run "Data Management/DataCleaning/2MergeColumn.py"

### 3.3 Step 3: Compressione Informativa e Standardizzazione Temporale
A valle della normalizzazione strutturale, il dataset presenta ridondanza informativa sotto forma di doppia quotazione bidirezionale (Bid e Ask). In questa fase, la pipeline si divide per consentire due approcci metodologici distinti alla rappresentazione del prezzo dell'asset, applicando contemporaneamente una standardizzazione ai formati temporali (adattamento ai formati `YYYY.MM.DD` e `HH:MM` tipici delle piattaforme di trading algoritmico).

* **Opzione A - Calcolo del Mid-Price (`3DoMedian.py`):** Questo script opera una sintesi delle quotazioni calcolando la media aritmetica (Mid-Price) tra le curve di Bid e Ask per ogni componente della candela (OHLC). Questo approccio neutralizza le fluttuazioni transitorie dello spread, fornendo un "Fair Value" continuo del sottostante, ideale per i modelli regressivi.
* **Opzione B - Estrazione Selettiva (`3_1DeleteAsk.py`):** In alternativa, questo script applica un filtro colonnare selettivo, isolando esclusivamente la curva dei prezzi Bid. Questa metrica riflette l'effettivo prezzo di liquidazione per posizioni long, risultando utile per simulazioni e metriche finanziarie orientate all'operatività reale.

**Nota Metodologica (Defensive Programming e Dati Sporchi):**
In aggiunta alle trasformazioni core, entrambi gli script implementano un livello di *Defensive Programming* essenziale per la gestione dei file estratti da provider finanziari. Durante l'iterazione di lettura, i parser verificano attivamente la presenza di stringhe testuali (es. la keyword "bid") all'interno dei record dati. Questo meccanismo di filtro permette di intercettare e scartare "intestazioni errate" o righe testuali finite accidentalmente nel mezzo del dataset. Prevenendo l'ingestione di queste righe sporche, si evitano crash fatali (eccezioni di cast a `float`) e si garantisce la continuità dell'elaborazione.

Entrambi gli script restituiscono un dataset snellito a 7 colonne (Date, Time, Open, High, Low, Close, TotalTicks), fungendo da input standardizzato per la successiva fase di pulizia statistica e gestione delle discontinuità (Gap temporali).

In [ ]:
# Esecuzione Step 3: Compressione Informativa e Standardizzazione Temporale con Media di prezzi tra Bid e Ask
%run "Data Management/DataCleaning/3DoMedian.py"

In [ ]:
# Esecuzione Step 3: Compressione Informativa e Standardizzazione Temporale eliminando i prezzi di Ask
%run "Data Management/DataCleaning/3_1DeleteAsk.py"

### 3.4 Step 4: Data Imputation, Risoluzione Discontinuità e Vettorizzazione (`4DataCleaning.py`)

Le serie storiche finanziarie, in particolare su timeframe ad alta frequenza (M1), sono fisiologicamente soggette a discontinuità. Queste possono essere classificate in interruzioni macroscopiche sistemiche (chiusure per weekend o festività) o micro-interruzioni anomale dovute a tick persi o assenza di liquidità. Per addestrare modelli basati su sequenze storiche continue, la totale assenza di discontinuità sull'asse temporale è un requisito fondamentale.

Il quarto step rappresenta il core della pipeline di preparazione dati. Rispetto all'approccio standard basato sul "Tempo di Calendario" (che forza la presenza temporale di tutti i giorni dell'anno solare), l'architettura software di questo modulo è stata evoluta per operare in puro **Tempo di Trading (Trading Time)**, eliminando il rischio di *Data Leakage* pulendo il dataset dai finti volumi del fine settimana.

Il processo è suddiviso in tre macro-fasi sequenziali:

1. **Taglio a Blocchi Operativi (Time-Series Segmentation):**
   Invece di forzare una reindicizzazione sull'intero range temporale disponibile, il sistema calcola i differenziali di tempo tra i tick isolando i macro-gap (es. interruzioni > 12 ore, tipiche di weekend e festività). Questi macro-gap vengono utilizzati come punti di "taglio" per segmentare il dataset in blocchi operativi continui.
   * **Vantaggio Metodologico:** La reindicizzazione a frequenza fissa (1 minuto) avviene esclusivamente all'interno di questi blocchi. In questo modo si evita l'iniezione artificiale di centinaia di migliaia di righe vuote (`NaN`) durante i weekend, azzerando l'impatto dei "Ghost Ticks" (finte quotazioni generate dai server dei broker a mercati chiusi o durante le fasi di manutenzione).

2. **Data Imputation Dinamica e Prevenzione del Data Leakage (Flat Doji):**
   La risoluzione dei micro-gap (i minuti mancanti all'interno di un blocco operativo attivo) avviene attraverso una logica di imputazione rigorosa:
   * **Abbandono dell'Interpolazione Lineare:** Nelle versioni iniziali si assumeva che l'assenza di tick per pochi minuti non alterasse il trend sottostante, creando una rampa di prezzo artificiale per unire due punti. Tuttavia, l'interpolazione introduce un grave *Data Leakage*, poiché calcola il prezzo di un minuto mancante utilizzando un dato del futuro (il prezzo di riapertura).
   * **Forward Fill (Flat Doji):** La nuova pipeline risolve tutti i gap intra-day propagando in avanti esclusivamente l'ultimo prezzo di chiusura noto. Vengono generate delle candele "piatte" (*Flat Doji*) in cui Open, High, Low e Close coincidono perfettamente con la chiusura precedente, assegnando un volume pari a zero. Questo approccio preserva l'integrità causale della serie storica, informando correttamente l'algoritmo che il mercato è rimasto illiquido o fermo, ed evitando la generazione di falsi micro-trend derivanti dall'interpolazione.

3. **Finalizzazione e Vettorizzazione (Export Ready):**
   Per minimizzare il collo di bottiglia dell'I/O (lettura/scrittura su disco di un dataset >500.000 righe), lo script elabora e mantiene in memoria entrambe le versioni del dataset (quella con Bid/Ask separati e quella unificata col Mid-Price):
   * Viene generato un dataset intermedio di appoggio (`FinalData.csv`) comprensivo di Date e Time formattati, utile per ispezioni umane, log e debug visivo.
   * Il dataframe in memoria subisce istantaneamente un *Feature Drop* per eliminare le variabili testuali ridondanti (Data e Ora stringa), estraendo esclusivamente l'indice `Datetime` e le variabili continue. Il risultato è un tensore puramente numerico (`XAUUSD_ReadyToUse.csv`), ottimizzato computazionalmente per l'ingestione diretta nell'algoritmo di similarità e nei modelli di Machine Learning.

In [ ]:
# Esecuzione Step 4: Vettorizzazione Dataframe
%run "Data Management/DataCleaning/4DataCleaning.py"

### 3.5 Automazione e Orchestrazione della Pipeline (`mainPipelineDataCleaning.py`)
Al fine di garantire la totale riproducibilità del processo di Data Preprocessing e preparare il sistema per un ambiente di test intensivo (come lo split cronologico out-of-sample e il successivo backtest quantitativo nel notebook 5), l'esecuzione sequenziale dei moduli è stata automatizzata tramite uno script orchestratore centralizzato.

L'approccio manuale "step-by-step", utile in fase di sviluppo e ispezione, è stato sostituito da un'architettura software di tipo *Pipeline*.
* **Meccanismo di Esecuzione:** Lo script sfrutta la libreria nativa `subprocess` per istanziare processi terminale indipendenti. I quattro nodi della pipeline (`1Converter.py`, `2MergeColumn.py`, lo step di compressione informativa e `4DataCleaning.py`) vengono richiamati in un rigoroso ordine cronologico.
* **Logging e Gestione degli Errori:** L'orchestratore implementa un sistema di cattura dell'output (sia `stdout` che `stderr`). In caso di successo, i log di ogni singolo modulo vengono consolidati e stampati a video per fornire un report di esecuzione. Qualora un nodo dovesse generare un'eccezione (es. file sorgente mancante o errore di parsing), la logica di *error handling* blocca immediatamente la pipeline, prevenendo l'elaborazione a cascata di dati corrotti.

Questa soluzione fornisce un singolo *entry point* di esecuzione ("One-Click Run") che trasforma i dati grezzi estratti dal broker nel tensore matematico `XAUUSD_ReadyToUse.csv`, azzerando il rischio di errori operativi umani e abbattendo i tempi di preparazione del dataset.

In [ ]:
# Esecuzione Pipeline Completa: Esegue tutti e 4 gli script (di default allo step 3 viene fatta la media dei prezzi)
%run "Data Management/DataCleaning/mainPipelineDataCleaning.py"